# 00 - Aircraft performance with Python: overview and conventions

This folder completes the performance course of the repository with the typical
calculations of **J. Roskam & C.T. Lan, *Airplane Aerodynamics and Performance***
(DARcorporation, 1997), cross-checked against **J.D. Anderson, *Aircraft
Performance and Design*** (McGraw-Hill, 1999). Both books are in `References/`.

| Notebook | Content | Roskam chapters |
|---|---|---|
| `00_Overview` | conventions, units, error handling, example aircraft | - |
| `01_AerodynamicDatabase_LevelFlight` | drag polar, characteristic points, stall, thrust/power required, max/min speed, flight envelope | 5, 8, 12.3 |
| `02_Range_Endurance_Loiter` | Breguet equations, numerical cruise, best speeds, wind, loiter, payload-range | 11 |
| `03_RateOfClimb` | rate of climb and climb angle, steep climbs, acceleration factor, ceilings, time to climb, OEI | 9 |
| `04_RateOfDescent` | gliding flight, hodograph, wind, idle descent, one-engine-inoperative drift-down | 8.2, 9.2.3, 9.3.3, 9.4.2 |
| `05_Takeoff_Landing` | take-off and landing distances (analytical, numerical, statistical), BFL | 10 |
| `06_Maneuvering_FlightEnvelope` | turns, load factors, V-n diagram | 12 |

## Design choices

* **The physics lives in a Python package, the explanation in the notebooks.**
  The package `aircraft_performance/` (repository root) contains small,
  documented functions - one per equation or method, each citing the Roskam
  equation number. The notebooks explain the theory, call the functions and
  discuss the results. The same code is exercised by an automated test suite
  (`tests/`, run with `pytest`) that reproduces the worked examples of both
  textbooks, so a mistake in a formula cannot go unnoticed.
* **SI units everywhere.** Data given in British or "inconsistent" units are
  converted *once*, with named constants (`u.FT`, `u.LBF`, `u.HP`, `u.KT`,
  `u.bsfc_to_si(...)`), never with magic numbers such as 603,500 or 53.5.
* **No silent failures.** Wrong inputs raise `InputError` immediately with a
  message naming the quantity; physically impossible requests (level flight
  above the ceiling, take-off with too little thrust...) raise
  `InfeasibleFlightConditionError`; results that are computed but questionable
  (a Mach number beyond drag divergence, a steep climb computed with small-angle
  formulas) emit a `PerformanceWarning`. Functions never return `NaN`.
* **Readability over speed.** Numerical methods are simple and explicit (grid
  search + Brent refinement, trapezoidal integration, `solve_ivp` for the
  ground roll).

In [1]:
# --- Setup -------------------------------------------------------------------
# The notebooks live in PrestazioniRoskam/, one level below the repository root,
# where the `aircraft_performance` package is. Make it importable:
import sys
import pathlib
ROOT = pathlib.Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

import aircraft_performance as ap
from aircraft_performance import units as u
from aircraft_performance.atmosphere import isa
from aircraft_performance.plotting import apply_style, reference_line
from aircraft_performance.reporting import print_table

apply_style()
print(f"aircraft_performance v{ap.__version__} ready")

aircraft_performance v1.0.0 ready


## 1. Units

Every function takes and returns SI quantities. The `units` module holds the
conversion factors: *multiply* to go to SI, *divide* to go back.
The specific fuel consumption deserves special care (see also
`Capitolo9/AutonomieMotoelica.ipynb`):

* propeller engines: fuel **weight** per unit shaft **energy**,
  $c\;[\mathrm{N/J}] = [1/\mathrm{m}]$;
* jet engines: fuel **weight** per unit **thrust** per unit time,
  $c_t\;[\mathrm{N/(N\,s)}] = [1/\mathrm{s}]$.

In [2]:
print(f"1 ft   = {u.FT} m")
print(f"1 lbf  = {u.LBF:.6f} N")
print(f"1 hp   = {u.HP:.3f} W")
print(f"1 kt   = {u.KT:.6f} m/s")
print()
c_bhp = 0.45                      # lb/(hp h), typical piston engine
c = u.bsfc_to_si(c_bhp)
print(f"BSFC = {c_bhp} lb/(hp h)  ->  c = {c:.4e} 1/m   (1/c = {1/c:,.0f} m)")
ct = u.tsfc_to_si(0.69)          # 1/h, typical turbofan
print(f"TSFC = 0.69 1/h          ->  c_t = {ct:.4e} 1/s")

1 ft   = 0.3048 m
1 lbf  = 4.448222 N
1 hp   = 745.700 W
1 kt   = 0.514444 m/s

BSFC = 0.45 lb/(hp h)  ->  c = 7.4565e-07 1/m   (1/c = 1,341,120 m)
TSFC = 0.69 1/h          ->  c_t = 1.9167e-04 1/s


## 2. Error handling in practice

The next cell deliberately makes three mistakes. Each one is caught and
reported with an explicit message instead of producing a wrong number.

In [3]:
from aircraft_performance.errors import InputError, InfeasibleFlightConditionError
from aircraft_performance import level_flight

examples = [
    ("negative wing area", lambda: ap.aerodynamics.stall_speed(10000.0, -16.0, 1.225, 1.6)),
    ("SFC given in lb/(hp h) instead of SI",
     lambda: ap.PropellerPropulsion(max_shaft_power_sl=120e3, propeller_efficiency=0.8, bsfc=0.45)),
    ("level flight at 9,000 m for a light piston aircraft",
     lambda: level_flight.level_flight_speeds(ap.examples.light_single_piston(), 9000.0)),
]
for label, action in examples:
    try:
        action()
    except (InputError, InfeasibleFlightConditionError) as err:
        print(f"[{type(err).__name__}] {label}:\n    {err}\n")

[InputError] negative wing area:
    'wing_area' must be strictly positive, got -16.0.

[InputError] SFC given in lb/(hp h) instead of SI:
    bsfc = 0.45 1/m is far too large: did you forget to convert from lb/(hp h)? Use units.bsfc_to_si().

[InfeasibleFlightConditionError] level flight at 9,000 m for a light piston aircraft:
    Level flight is impossible at h = 9000 m, W = 10787 N: the maximum excess thrust is -393.7 N < 0 (above the absolute ceiling).



## 3. The example aircraft

Four aircraft are used throughout the notebooks. Their data are
representative (patterned after the textbook examples), not certified data:

* **Light single-engine piston** - the aircraft of the `Capitolo9` exercises
  (1,100 kg, 16 m$^2$, $C_{D_0}=0.030$, $e=0.8$, 160 hp);
* **Roskam light twin** - the Cessna 310-like twin of Roskam Examples 9.1,
  10.2 and 10.3;
* **Regional turboprop** - ATR 72-like, as in the `Capitolo9` ATR exercise;
* **Business jet** - the Gulfstream IV-like jet used throughout Anderson,
  Chapters 5-6 ($C_D = 0.015 + 0.08\,C_L^2$, $T = 2\times13{,}850$ lb).

An `Aircraft` is *immutable*: to study a variant use `aircraft.replace(...)`,
which returns a modified copy and leaves the original untouched.

In [4]:
for name, aircraft in ap.examples.all_examples().items():
    print(aircraft.summary())
    print()

Aircraft: Light single-engine piston (C172-like)
  mass            =     1100.0 kg   (W = 10.79 kN)
  wing area S     =      16.00 m^2
  span b          =      11.00 m    (AR = 7.56)
  wing loading    =      674.2 N/m^2
  [clean   ] CD = 0.0300 + 0.0526 CL^2,  CL_max = 1.60,  E_max = 12.59
  [takeoff ] CD = 0.0400 + 0.0526 CL^2,  CL_max = 1.90,  E_max = 10.90
  [landing ] CD = 0.0700 + 0.0526 CL^2,  CL_max = 2.10,  E_max = 8.24
  propeller: P_SL = 119.3 kW (1 engines), eta_p = 0.75, BSFC = 7.456e-07 1/m, lapse 'gagg_ferrar'

Aircraft: Roskam light twin (C310-like)
  mass            =     2086.5 kg   (W = 20.46 kN)
  wing area S     =      16.26 m^2
  span b          =      10.67 m    (AR = 7.00)
  wing loading    =     1258.6 N/m^2
  [clean   ] CD = 0.0293 + 0.0557 CL^2,  CL_max = 1.31,  E_max = 12.38
  [takeoff ] CD = 0.0620 + 0.0568 CL^2,  CL_max = 1.69,  E_max = 8.42
  [landing ] CD = 0.1000 + 0.0568 CL^2,  CL_max = 2.12,  E_max = 6.63
  propeller: P_SL = 387.8 kW (2 engines), eta_p

In [5]:
# Example: build your own aircraft in a few lines
my_polar = ap.ParabolicDragPolar.from_aspect_ratio(CD0=0.027, aspect_ratio=8.0, oswald_factor=0.8)
my_aircraft = ap.Aircraft(
    name="My trainer",
    mass=1200.0, wing_area=15.0, wing_span=11.0,
    configurations={"clean": ap.Configuration(my_polar, CL_max=1.5)},
    propulsion=ap.PropellerPropulsion(max_shaft_power_sl=180 * u.HP, propeller_efficiency=0.78,
                                      bsfc=u.bsfc_to_si(0.46), static_thrust=3000.0),
)
print(my_aircraft.summary())
heavier = my_aircraft.replace(mass=1300.0)
print(f"\noriginal mass: {my_aircraft.mass} kg, modified copy: {heavier.mass} kg")

Aircraft: My trainer
  mass            =     1200.0 kg   (W = 11.77 kN)
  wing area S     =      15.00 m^2
  span b          =      11.00 m    (AR = 8.07)
  wing loading    =      784.5 N/m^2
  [clean   ] CD = 0.0270 + 0.0497 CL^2,  CL_max = 1.50,  E_max = 13.64
  propeller: P_SL = 134.2 kW (1 engines), eta_p = 0.78, BSFC = 7.622e-07 1/m, lapse 'gagg_ferrar'

original mass: 1200.0 kg, modified copy: 1300.0 kg


## 4. Running the tests

From the repository root:

```bash
pip install -r requirements.txt
python -m pytest            # ~100 tests, including the textbook examples
```

The tests are also the best place to see *how much* the results agree with the
books: for instance `tests/test_level_climb_descent.py` reproduces Anderson's
table of maximum rate of climb versus altitude for the Gulfstream IV, and
`tests/test_range_field_maneuver.py` reproduces Roskam Examples 10.2, 10.3, 11.1-11.3.